# Wikipedia RAG — Question & Answer UI
Run **all cells once**, then type questions into the interface at the bottom. This notebook reuses the completed full Wikipedia BM25 index and local Flan-T5-small model. It does not build an index, download assets, or run ablations.

Each click retrieves real passages from SQLite, sends that evidence through Flan-T5-small, and shows the answer, supporting citations, and all retrieved passages. Concatenate is the faster default. Generate-from-evidence makes up to k+1 model calls. Answers can take a minute or more on this corpus. Wait for the status to change to complete before submitting another question.

Use the same Python kernel as the full notebook. If widgets are missing, install `ipywidgets` in that kernel while online (`%pip install ipywidgets`), restart the kernel, and rerun. JupyterLab, Notebook and VS Code must have widget rendering enabled. After a kernel restart, run all cells again. No API key is required.


In [21]:
# Load Arrow first to match the working full-notebook runtime on this environment.
import pyarrow
from pathlib import Path
import os, re, json, sqlite3, time, html
from urllib.parse import urlparse
from IPython.display import display, HTML, clear_output
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("Install ipywidgets in this kernel, restart, then rerun this notebook.") from exc

ROOT = Path('/Users/amolsingh/Documents/code/IITB-DL')
INDEX_PATH = ROOT / 'data/full_wikipedia_bm25.sqlite'
MODEL_DIR = ROOT / 'models/flan-t5-small'
DEVICE = 'cpu'
THREADS = 4


In [22]:
def offline_mode() -> None:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [23]:
UNKNOWN = "I don't know based on the provided evidence."
WORD = re.compile(r"\w+", re.UNICODE)
STOPWORDS = set(
    "a an the and or but of to in on at for from by with as is are was were be "
    "been being do does did has have had what which who whom whose when where "
    "why how can could would should will may might that this these those it its "
    "into about according many much name tell me give".split()
)


In [24]:
def span(text: str, candidate: str) -> str | None:
    """Return an original-text span, matching complete casefolded word tokens."""
    needles = [m.group().casefold() for m in WORD.finditer(candidate)]
    if not needles:
        return None
    matches = list(WORD.finditer(text))
    haystack = [m.group().casefold() for m in matches]
    for i in range(len(haystack) - len(needles) + 1):
        if haystack[i : i + len(needles)] == needles:
            return text[matches[i].start() : matches[i + len(needles) - 1].end()]
    return None

def is_unknown(text: str) -> bool:
    normalized = " ".join(WORD.findall(text.casefold()))
    return not normalized or normalized in {
        "unknown", "unanswerable", "not enough information", "not provided",
        "i don t know", "i do not know", "no answer", "not in the context",
        "i don t know based on the provided evidence",
    }

def passages(article: dict, chunk_words: int = 100, overlap: int = 25):
    if chunk_words < 1 or overlap < 0 or overlap >= chunk_words:
        raise ValueError("Require chunk_words > overlap >= 0.")
    text = article["text"]
    words = list(re.finditer(r"\S+", text))
    for ordinal, start in enumerate(range(0, len(words), chunk_words - overlap)):
        end = min(start + chunk_words, len(words))
        char_start, char_end = words[start].start(), words[end - 1].end()
        yield {
            "passage_id": f'{article["id"]}:{ordinal}',
            "article_id": str(article["id"]),
            "title": article["title"],
            "url": article["url"],
            "text": text[char_start:char_end],
            "char_start": char_start,
            "char_end": char_end,
        }
        if end == len(words):
            break


In [25]:
class Retriever:
    def __init__(self, path: Path):
        if not path.is_file():
            raise FileNotFoundError(f"Index missing: {path}. Run the index command first.")
        self.connection = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)
        self.connection.row_factory = sqlite3.Row
        self.metadata = {
            row["key"]: json.loads(row["value"])
            for row in self.connection.execute("SELECT key,value FROM metadata")
        }

        if not self.metadata.get('complete'):
            self.connection.close()
            raise RuntimeError('Full corpus build is incomplete. Resume the indexing cell.')

    def close(self):
        self.connection.close()

    def search(self, question: str, k: int = 3) -> list[dict]:
        if k < 0:
            raise ValueError("k must be nonnegative.")
        terms = list(dict.fromkeys(
            token.casefold() for token in WORD.findall(question)
            if token.casefold() not in STOPWORDS
        ))[:48]
        if not terms or k == 0:
            return []
        # Quoted tokens and bound parameters prevent FTS query syntax injection.
        query = " OR ".join(f'"{term}"' for term in terms)
        rows = self.connection.execute("""
            SELECT p.*, bm25(search,2.0,1.0) AS bm25
            FROM search JOIN passages p ON p.rowid=search.rowid
            WHERE search MATCH ? ORDER BY bm25 ASC, p.rowid ASC LIMIT ?
        """, (query, k))
        return [dict(row) for row in rows]


In [26]:
class Generator:
    def __init__(self, model_dir: Path | None = None, device="cpu", threads=4,
                 max_input_tokens=512, max_new_tokens=48):
        offline_mode()
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        model_dir = model_dir or ROOT / "models" / "flan-t5-small"
        if not (model_dir / "model.safetensors").exists():
            raise FileNotFoundError(f"Model missing at {model_dir}. Run prepare while online.")
        if threads < 1 or max_new_tokens < 1 or not 64 <= max_input_tokens <= 512:
            raise ValueError("Require positive threads/output tokens and input budget 64..512.")
        torch.set_num_threads(threads)
        torch.manual_seed(42)
        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_dir, local_files_only=True, use_safetensors=True,
        ).to(device).eval()
        self.device = device
        self.max_input_tokens = max_input_tokens
        self.max_new_tokens = max_new_tokens
        self.threads = threads

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=True, verbose=False))

    @staticmethod
    def render(question: str, evidence: list[dict], stage="answer") -> str:
        if stage == "closed_book":
            return f"Answer the question briefly.\nQuestion: {question}\nAnswer:"
        instruction = (
            "Copy the shortest phrase from the context that answers the question."
            if stage == "extract"
            else "Answer the question briefly using only the context."
        )
        context = "\n\n".join(
            f'[{item["passage_id"]}] {item["title"]}\n{item["text"]}' for item in evidence
        )
        return (
            f"{instruction} If the context does not contain the answer, say unknown.\n"
            f"Context:\n{context}\nQuestion: {question}\nAnswer:"
        )

    def pack(self, question: str, evidence: list[dict], stage="answer"):
        """Fairly allocate a fixed input budget, preserving whole words and headers."""
        question = question.strip()
        if not question:
            raise ValueError("Question cannot be empty.")
        words = [re.findall(r"\S+", item["text"]) for item in evidence]

        def make(lengths):
            visible = [
                {key: item[key] for key in ("passage_id", "title", "url")}
                | {"text": " ".join(tokens[:length])}
                for item, tokens, length in zip(evidence, words, lengths)
            ]
            prompt = self.render(question, visible, stage)
            return prompt, visible, self.count_tokens(prompt)

        if not evidence:
            prompt, visible, size = make([])
            if size > self.max_input_tokens:
                raise ValueError("Question exceeds the model input budget.")
            return prompt, visible, size
        _, _, overhead = make([0] * len(words))
        if overhead > self.max_input_tokens - 16:
            raise ValueError("Question/source headers leave too little evidence space; reduce k.")
        low, high = 0, max(map(len, words))
        while low < high:
            middle = (low + high + 1) // 2
            if make([min(middle, len(tokens)) for tokens in words])[2] <= self.max_input_tokens:
                low = middle
            else:
                high = middle - 1
        lengths = [min(low, len(tokens)) for tokens in words]
        # Fill remaining capacity in retrieval order, never truncate the prompt.
        changed = True
        while changed:
            changed = False
            for i, tokens in enumerate(words):
                if lengths[i] < len(tokens):
                    trial = lengths.copy()
                    trial[i] += 1
                    if make(trial)[2] <= self.max_input_tokens:
                        lengths = trial
                        changed = True
        return make(lengths)

    def generate(self, prompt: str) -> dict:
        encoded = self.tokenizer(prompt, return_tensors="pt", truncation=False).to(self.device)
        input_tokens = int(encoded.input_ids.shape[1])
        if input_tokens > self.max_input_tokens:
            raise ValueError(f"Input has {input_tokens} tokens; budget is {self.max_input_tokens}.")
        start = time.perf_counter()
        with self.torch.inference_mode():
            output = self.model.generate(
                **encoded, do_sample=False, num_beams=1,
                max_new_tokens=self.max_new_tokens,
            )
        elapsed = time.perf_counter() - start
        return {
            "text": self.tokenizer.decode(output[0], skip_special_tokens=True).strip(),
            "input_tokens": input_tokens, "output_tokens": int(output.shape[1] - 1),
            "generation_seconds": elapsed,
        }


In [27]:
def supporting_sources(answer: str, visible: list[dict], originals: list[dict]) -> list[dict]:
    if is_unknown(answer):
        return []
    lookup = {item["passage_id"]: item for item in originals}
    sources = []
    for item in visible:
        original = lookup[item["passage_id"]]
        # Check both the model-visible excerpt and the original passage.
        if span(item["text"], answer) is not None:
            quote = span(original["text"], answer)
            if quote is not None:
                sources.append({
                    "passage_id": item["passage_id"], "title": original["title"],
                    "url": original["url"], "answer_span": quote,
                    "visible_excerpt": item["text"],
                    "char_start": original["char_start"], "char_end": original["char_end"],
                })
    return sources

def evidence_sentence(text: str, candidate: str) -> str:
    """Keep the candidate's original sentence so the reduce stage retains its relation."""
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if span(sentence, candidate) is not None:
            return sentence.strip()
    # A generated span can cross sentence boundaries; retain the visible passage.
    return text


In [28]:
def answer_question(generator: Generator, question: str, retrieved: list[dict],
                    method="concatenate") -> dict:
    if method not in {"concatenate", "generate_from_evidence", "closed_book"}:
        raise ValueError(f"Unknown method: {method}")
    start = time.perf_counter()
    trace, map_steps = [], []
    if method == "closed_book":
        prompt, visible, _ = generator.pack(question, [], stage="closed_book")
        generated = generator.generate(prompt)
        trace.append({"stage": "closed_book", "prompt": prompt, **generated})
        raw_answer = generated["text"]
        sources = []
        answer = UNKNOWN if is_unknown(raw_answer) else raw_answer
    elif not retrieved:
        raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    else:
        evidence = retrieved
        if method == "generate_from_evidence":
            evidence = []
            for passage in retrieved:
                prompt, map_visible, _ = generator.pack(question, [passage], stage="extract")
                generated = generator.generate(prompt)
                trace.append({"stage": "extract", "prompt": prompt, **generated})
                candidate = generated["text"]
                supported = supporting_sources(candidate, map_visible, [passage])
                map_steps.append({
                    "passage_id": passage["passage_id"], "candidate": candidate,
                    "accepted": bool(supported), "visible_evidence": map_visible,
                })
                if supported:
                    # The reduce stage receives original quotes, never unverified model claims.
                    quote = evidence_sentence(map_visible[0]["text"], candidate)
                    map_steps[-1]["evidence_sentence"] = quote
                    evidence.append(passage | {"text": quote})
        if evidence:
            prompt, visible, _ = generator.pack(question, evidence)
            generated = generator.generate(prompt)
            trace.append({"stage": "answer", "prompt": prompt, **generated})
            raw_answer = generated["text"]
            sources = supporting_sources(raw_answer, visible, retrieved)
            answer = raw_answer if sources else UNKNOWN
        else:
            raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    return {
        "question": question, "method": method, "requested_passages_used": len(retrieved),
        "answer": answer, "raw_answer": raw_answer, "abstained": answer == UNKNOWN,
        "span_supported": bool(sources), "citations": sources, "retrieved": retrieved,
        "visible_evidence": visible, "map_steps": map_steps, "trace": trace,
        "generation_calls": len(trace),
        "total_input_tokens": sum(item["input_tokens"] for item in trace),
        "total_output_tokens": sum(item["output_tokens"] for item in trace),
        "latency_seconds": time.perf_counter() - start,
        "grounding_policy": "contiguous lexical span in visible and original evidence"
        if method != "closed_book" else "none (closed-book baseline)",
    }


In [29]:
# Reuse loaded assets when rerunning this cell in the same kernel.
if 'ui_retriever' not in globals():
    ui_retriever = Retriever(INDEX_PATH)
if 'ui_generator' not in globals():
    ui_generator = Generator(MODEL_DIR, device=DEVICE, threads=THREADS)
print('Ready: completed full Wikipedia index and local Flan-T5-small loaded.')


Ready: completed full Wikipedia index and local Flan-T5-small loaded.


In [ ]:
def safe_link(url):
    return html.escape(url, quote=True) if urlparse(url).scheme in {'http','https'} else '#'

def render_result(result, retrieval_seconds):
    esc = html.escape
    answer = esc(result['answer'])
    status = 'No supported answer found' if result['abstained'] else 'Answer'
    parts = [f'<h3>{status}</h3><p style="font-size:1.15em">{answer}</p>']
    parts.append(f"<p>Retrieved {len(result['retrieved'])} passages in {retrieval_seconds:.1f}s · "
                 f"Answer pipeline: {result['latency_seconds']:.1f}s · "
                 f"Model calls: {result['generation_calls']}</p>")
    if result['abstained']:
        parts.append('<p>Try a more specific question or different retrieval size. Relevant information may still exist in Wikipedia.</p>')
    if result['citations']:
        parts.append('<h4>Supporting evidence</h4>')
        for source in result['citations']:
            parts.append(f'<p><a href="{safe_link(source["url"])}" target="_blank" rel="noopener noreferrer">'
                         f'{esc(source["title"])}</a> — {esc(source["passage_id"])}</p>'
                         f'<blockquote>{esc(source["visible_excerpt"])}</blockquote>')
    parts.append('<details><summary>Inspect all retrieved passages</summary>')
    visible_ids = {p['passage_id'] for p in result['visible_evidence']}
    for rank,p in enumerate(result['retrieved'],1):
        parts.append(f'<h4>{rank}. {esc(p["title"])}</h4><p>{esc(p["text"])}</p>'
                     f'<small>Passage {esc(p["passage_id"])} · '
                     f'Included in final prompt: {p["passage_id"] in visible_ids}</small>')
    parts.append('</details><p><small>Source-span checks provide lexical support; they do not guarantee factual correctness.</small></p>')
    return ''.join(parts)

# Reexecuting this cell replaces the previous interface rather than leaving duplicate callbacks.
if 'qa_panel' in globals():
    qa_panel.close()
question_box = widgets.Textarea(placeholder='Example: What is the atomic number of aluminium?',
    description='Question:', layout=widgets.Layout(width='95%',height='85px'))
k_picker = widgets.Dropdown(options=[1,3,5],value=3,description='Passages:')
method_picker = widgets.Dropdown(options=[('Concatenate (faster)','concatenate'),
    ('Generate from evidence','generate_from_evidence')],value='concatenate',description='Method:',
    layout=widgets.Layout(width='360px'))
ask_button = widgets.Button(description='Ask Wikipedia',button_style='primary',icon='search')
clear_button = widgets.Button(description='Clear')
status_label = widgets.HTML(value='Ready. Ask a factual question about the Wikipedia snapshot.')
answer_output = widgets.Output()
last_rag_result = None

def on_ask(_):
    global last_rag_result
    question = question_box.value.strip()
    if not question:
        status_label.value = 'Please enter a question.'
        return
    for control in [ask_button,clear_button,question_box,k_picker,method_picker]:
        control.disabled = True
    answer_output.clear_output(wait=True)
    status_label.value = '<b>Retrieving passages from the full Wikipedia index…</b>'
    try:
        start = time.perf_counter()
        passages_found = ui_retriever.search(question,k_picker.value)
        retrieval_seconds = time.perf_counter()-start
        status_label.value = '<b>Generating an answer from retrieved evidence…</b>'
        last_rag_result = answer_question(ui_generator,question,passages_found,method_picker.value)
        with answer_output:
            display(HTML(render_result(last_rag_result,retrieval_seconds)))
        status_label.value = '<b>Complete.</b> You can ask another question.'
    except Exception as exc:
        last_rag_result = None
        status_label.value = '<b>Could not complete the request.</b>'
        with answer_output:
            print(f'{type(exc).__name__}: {exc}')
    finally:
        for control in [ask_button,clear_button,question_box,k_picker,method_picker]:
            control.disabled = False

def on_clear(_):
    global last_rag_result
    question_box.value = ''
    last_rag_result = None
    answer_output.clear_output()
    status_label.value = 'Ready.'

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)
qa_panel = widgets.VBox([widgets.HTML('<h2>Ask Wikipedia</h2><p>Offline retrieval + Flan-T5-small</p>'),
    question_box,widgets.HBox([k_picker,method_picker]),widgets.HBox([ask_button,clear_button]),
    status_label,answer_output])
display(qa_panel)
